## News Stream Synthetic Data Create —
Fine-Tuning Qwen3-4B with Unsloth (QLoRA)
### Save it in Collab
**Runtime:** Colab → change runtime type → **T4 GPU** (free tier is enough for a 4B model with QLoRA).

This notebook does five things:
1. Install Unsloth and load **Qwen3-4B** in 4-bit (QLoRA).
2. Attach LoRA adapters (the tiny trainable side-matrices).
3. **Create** your `news_stream_400_train.jsonl` / `_val.jsonl`


## 1. Install
Unsloth pins compatible versions of transformers/trl/peft for you. Pin Unsloth itself if you want reproducibility across class sessions — APIs move fast.

In [7]:
# %%capture
!uv pip install -U unsloth unsloth_zoo
# If a class needs a frozen, reproducible setup, pin instead, e.g.:
# !pip install "unsloth==2026.5.*" "unsloth_zoo==2026.5.*"


Using Python 3.13.15 environment at: /usr
Resolved 101 packages in 267ms
Checked 101 packages in 2ms


### Restart Sessions (Control + M .)
so numpy package mismatch will be taken care of

## 2. Bring in your dataset

Upload `news_stream_400_train.jsonl` and `news_stream_400_val.jsonl` (left sidebar → Files → upload), or pull them from Google Drive / HF. Each line is already in the `messages` format:


In [9]:
from datasets import load_dataset

datasets = load_dataset("json", data_files="combined_43words_Summary.jsonl", split="train")
# val_dataset = load_dataset("json", data_files="multilingual_dataset.jsonl", split="val")

print(datasets)
# print("example:", train_dataset[0][0])   # a user turn
# print("Valexpample:", val_dataset[0]["messages"][1])   # a user turn


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['language', 'split', 'text', 'summary'],
    num_rows: 1504
})


In [2]:
# from datasets import load_dataset

# languages = [
#     "odia",
#     "hindi",
#     "english",
#     "telugu",
#     "bengali",
#     "tamil",
# ]

# datasets = {}

# for lang in languages:
#     dataset_name = f"{lang}-{lang}"

#     datasets[lang] = load_dataset(
#         "csv",
#         data_files={
#             "train": f"hf://datasets/PMIndiaData/PMIndiaSum/{dataset_name}/train.csv",
#             "validation": f"hf://datasets/PMIndiaData/PMIndiaSum/{dataset_name}/valid.csv",
#             "test": f"hf://datasets/PMIndiaData/PMIndiaSum/{dataset_name}/test.csv",
#         }
#     )

#     print(
#         lang,
#         len(datasets[lang]["train"]),
#         len(datasets[lang]["validation"]),
#         len(datasets[lang]["test"])
#     )

In [10]:
print(datasets)

Dataset({
    features: ['language', 'split', 'text', 'summary'],
    num_rows: 1504
})


In [4]:
# import json

# output_file = "multilingual_dataset.jsonl"

# with open(output_file, "w", encoding="utf-8") as f:
#     for language, dataset_dict in datasets.items():
#         for split, dataset in dataset_dict.items():
#             for row in dataset:
#                 record = {
#                     "language": language,
#                     "split": split,
#                     "source_url": row["source_url"],
#                     "target_url": row["target_url"],
#                     "text": row["text"],
#                     "summary": row["summary"],
#                 }

#                 f.write(json.dumps(record, ensure_ascii=False) + "\n")

# print(f"Saved to: {output_file}")

In [5]:
# import json
# import random

# random.seed(42)

# for language, dataset_dict in datasets.items():

#     output_file = f"{language}_444.jsonl"

#     all_records = []

#     for split, dataset in dataset_dict.items():
#         for row in dataset:
#             all_records.append({
#                 "language": language,
#                 "split": split,
#                 # "source_url": row["source_url"],
#                 # "target_url": row["target_url"],
#                 "text": row["text"],
#                 "summary": row["summary"],
#             })

#     selected_records = random.sample(
#         all_records,
#         min(444, len(all_records))
#     )

#     with open(output_file, "w", encoding="utf-8") as f:
#         for record in selected_records:
#             f.write(
#                 json.dumps(record, ensure_ascii=False) + "\n"
#             )

#     print(f"{language}: saved {len(selected_records)} records → {output_file}")

In [6]:
# def prepare_dataset(dataset, language):

#     dataset = dataset.add_column(
#         "language",
#         [language] * len(dataset)
#     )

#     return dataset
# for lang in languages:

#     datasets[lang]["train"] = prepare_dataset(
#         datasets[lang]["train"],
#         lang
#     )

#     datasets[lang]["validation"] = prepare_dataset(
#         datasets[lang]["validation"],
#         lang
#     )

#     datasets[lang]["test"] = prepare_dataset(
#         datasets[lang]["test"],
#         lang
#     )

In [7]:
# from datasets import concatenate_datasets

# dataset = concatenate_datasets([
#     datasets["english"]["train"],
#     datasets["odia"]["train"],
#     datasets["hindi"]["train"],
#     datasets["telugu"]["train"],
#     datasets["tamil"]["train"],
#     datasets["bengali"]["train"],
# ])

ValueError: Column 'english' doesn't exist.

In [9]:
# valid_dataset = concatenate_datasets([
#     datasets["english"]["validation"],
#     datasets["odia"]["validation"],
#     datasets["hindi"]["validation"],
#     datasets["telugu"]["validation"],
#     datasets["tamil"]["validation"],
#     datasets["bengali"]["validation"],
# ])

In [10]:
# test_dataset = concatenate_datasets([
#     datasets["english"]["test"],
#     datasets["odia"]["test"],
#     datasets["hindi"]["test"],
#     datasets["telugu"]["test"],
#     datasets["tamil"]["test"],
#     datasets["bengali"]["test"],
# ])

In [11]:
from collections import Counter

print("TRAIN")
print(Counter(datasets["language"]))

# print("\nVALIDATION")
# print(Counter(valid_dataset["language"]))

# print("\nTEST")
# print(Counter(test_dataset["language"]))

TRAIN
Counter({'hindi': 487, 'odia': 487, 'english': 487, 'telugu': 43})


### Good Luck for Fine Tuning